*(Original request: explain conditional edge routing — see `2.0-langgraph-core-concepts.ipynb` for that content. This notebook covers post-1.0 features.)*

# LangGraph Post-1.0 Features

This notebook surveys features added to LangGraph after the 1.0 release. Each section has a minimal runnable example.

In [ ]:
# %pip install langgraph langchain-openai

## The Command Primitive

Command combines state update + routing in a single return. Replaces the dict-return + separate conditional edge pattern.

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command
from typing import TypedDict, Literal

class State(TypedDict):
    value: int
    route: str

def decide(state: State) -> Command[Literal["path_a", "path_b"]]:
    if state["value"] > 10:
        return Command(update={"route": "A"}, goto="path_a")
    return Command(update={"route": "B"}, goto="path_b")

def path_a(state: State): return {"route": "A done"}
def path_b(state: State): return {"route": "B done"}

g = StateGraph(State)
g.add_node("decide", decide)
g.add_node("path_a", path_a)
g.add_node("path_b", path_b)
g.add_edge(START, "decide")
g.add_edge("path_a", END)
g.add_edge("path_b", END)
graph = g.compile()
print(graph.invoke({"value": 15, "route": ""}))
print(graph.invoke({"value": 5, "route": ""}))

# Subgraphs in LangGraph

Subgraphs let you compose graphs inside other graphs. They are the primary mechanism for building **modular, multi-agent systems** in LangGraph — each agent (or team of agents) lives in its own subgraph, and a parent graph orchestrates how they collaborate.

In [ ]:
# %pip install langgraph

## What are Subgraphs?

A **subgraph** is just a compiled LangGraph that you embed inside another graph. There are two main reasons to use them:

1. **Composition** — build complex graphs out of smaller, self-contained pieces
2. **Multi-agent systems** — each agent is a subgraph; the parent graph routes between them

There are two patterns for embedding a subgraph, depending on whether it shares state keys with the parent or has its own private schema.

## Pattern 1 — Shared State Keys

When the subgraph's state schema **shares keys** with the parent graph, you can add the compiled subgraph **directly as a node**. LangGraph will automatically pass the shared keys in and merge the output keys back out.

This is the simplest and most common pattern.

In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

# Subgraph state (shares 'messages' key with parent)
class SubgraphState(TypedDict):
    messages: list
    sub_result: str

def sub_node(state: SubgraphState):
    last = state["messages"][-1] if state["messages"] else "none"
    return {"sub_result": f"Processed: {last}"}

sub_builder = StateGraph(SubgraphState)
sub_builder.add_node("sub_node", sub_node)
sub_builder.add_edge(START, "sub_node")
sub_builder.add_edge("sub_node", END)
subgraph = sub_builder.compile()

# Parent graph
class ParentState(TypedDict):
    messages: list
    sub_result: str
    final: str

def finalize(state: ParentState):
    return {"final": f"Done: {state['sub_result']}"}

parent_builder = StateGraph(ParentState)
parent_builder.add_node("subgraph", subgraph)   # subgraph added directly as node
parent_builder.add_node("finalize", finalize)
parent_builder.add_edge(START, "subgraph")
parent_builder.add_edge("subgraph", "finalize")
parent_builder.add_edge("finalize", END)
parent = parent_builder.compile()

result = parent.invoke({"messages": ["hello"], "sub_result": "", "final": ""})
print(result)

## Pattern 2 — Different Schemas

When the subgraph has a **completely different state schema** from the parent, you cannot add it directly. Instead, wrap it inside a regular node function that:

1. Translates parent state → subgraph input
2. Invokes the subgraph
3. Translates subgraph output → parent state update

This pattern is useful when you want a subgraph to keep its internals **private** — only exposing a clean input/output contract to the parent.

In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

# Subgraph with its own private state
class AnalysisState(TypedDict):
    text: str
    word_count: int

def count_words(state: AnalysisState):
    return {"word_count": len(state["text"].split())}

analysis_builder = StateGraph(AnalysisState)
analysis_builder.add_node("count_words", count_words)
analysis_builder.add_edge(START, "count_words")
analysis_builder.add_edge("count_words", END)
analysis_subgraph = analysis_builder.compile()

# Parent graph with different schema — invoke subgraph inside a node
class MainState(TypedDict):
    content: str
    analysis: dict

def run_analysis(state: MainState):
    # Translate parent state → subgraph state
    sub_result = analysis_subgraph.invoke({"text": state["content"], "word_count": 0})
    return {"analysis": sub_result}

main_builder = StateGraph(MainState)
main_builder.add_node("run_analysis", run_analysis)
main_builder.add_edge(START, "run_analysis")
main_builder.add_edge("run_analysis", END)
main_graph = main_builder.compile()

result = main_graph.invoke({"content": "Hello world this is LangGraph", "analysis": {}})
print(result)

## Persistence in Subgraphs

**Per-thread persistence is the default and recommended approach.** When you compile the parent graph with a checkpointer, every subgraph invocation within that thread is automatically checkpointed too — no extra wiring needed.

Per-invocation persistence (where state is reset on every call) is rarely useful for stateful multi-agent systems. Use per-thread persistence when you want subgraph agents to remember context across turns of a conversation.

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

# Compile the parent graph (from Pattern 1) with a checkpointer
memory = MemorySaver()
checkpointed_parent = parent_builder.compile(checkpointer=memory)

config = {"configurable": {"thread_id": "agent-thread-1"}}
result = checkpointed_parent.invoke(
    {"messages": ["hello"], "sub_result": "", "final": ""},
    config=config,
)
print(result)

# Subgraph state is persisted as part of the parent thread
state = checkpointed_parent.get_state(config)
print("Persisted state:", state.values)

## When to Use Subgraphs

Reach for subgraphs when you need:

- **Complex multi-agent teams** — each agent or team is its own subgraph, the parent orchestrates
- **Reusable agent modules** — package a subgraph once, drop it into many parent graphs
- **Isolating failure domains** — a crash inside a subgraph node is easier to reason about than a crash inside a giant monolithic graph
- **Different teams owning different components** — each team ships and tests its subgraph independently

If your graph is small and the logic is linear, you probably don't need subgraphs yet. Reach for them when complexity demands modularity.

## Functional API

The Functional API (`@entrypoint`, `@task`) lets you write graphs as regular Python functions with decorators instead of explicit node/edge wiring. Released in LangGraph 0.3+; useful for simpler graphs.

In [ ]:
from langgraph.func import entrypoint, task
from langgraph.checkpoint.memory import MemorySaver

@task
def fetch_data(query: str) -> str:
    # In a real app, call a search API here
    return f"Results for: {query}"

@task
def summarize(data: str) -> str:
    return f"Summary: {data[:50]}..."

@entrypoint(checkpointer=MemorySaver())
def research_pipeline(query: str) -> str:
    data = fetch_data(query).result()
    summary = summarize(data).result()
    return summary

# Invoke like a regular graph
config = {"configurable": {"thread_id": "func-1"}}
result = research_pipeline.invoke("LangGraph features", config=config)
print(result)

## MCP Integration with `langchain-mcp-adapters`

The `langchain-mcp-adapters` package connects LangGraph agents to any Model Context Protocol (MCP) server — converting MCP tools to LangChain tools.

```bash
pip install langchain-mcp-adapters langgraph "langchain[openai]"
```

MCP servers expose tools via two transports:
- **stdio**: Local process (most common for local tools)
- **streamable HTTP with SSE fallback**: Remote servers

See: https://github.com/langchain-ai/langchain-mcp-adapters

In [ ]:
# MCP (Model Context Protocol) — connect an agent to a local MCP server via stdio
#
# my_mcp_server.py exposes three tools: get_weather, calculate, lookup_fact
# Run this cell as-is — the server is started automatically as a subprocess.

import asyncio
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.prebuilt import create_react_agent
from langchain_openai import ChatOpenAI

client = MultiServerMCPClient(
    {
        "my_server": {
            "command": "python",
            "args": ["my_mcp_server.py"],
            "transport": "stdio",
        }
    }
)

tools = await client.get_tools()
print("Tools loaded from MCP server:", [t.name for t in tools])

model = ChatOpenAI(model="gpt-4o-mini")
agent = create_react_agent(model, tools)

result = await agent.ainvoke(
    {"messages": [{"role": "user", "content": "What's the weather in Lisbon? Also, what is 42 * 7?"}]}
)
print(result["messages"][-1].content)


## Durable Execution

LangGraph Platform provides **durable execution** — if a run is interrupted (network error, timeout, server restart), it can be resumed from the last checkpoint. This is built on the same checkpointer system covered in `5.0-persistence-and-checkpointers.ipynb`. In production deployments, combine a SqliteSaver or PostgresSaver with the `langgraph deploy` command to get durable, resumable runs automatically.